<a href="https://colab.research.google.com/github/playmich2018-coder/asr-whisper-fastapi/blob/main/Caso_Pr%C3%A1ctico3_Question_Answering_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Paso 1: Instalación de Librerías Modernas

In [1]:
!pip install -q transformers datasets evaluate accelerate

Paso 2: Carga Eficiente del Modelo y los Datos

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

# Agregamos 'rajpurkar/' para cumplir con el formato namespace/name exigido
datasets = load_dataset("rajpurkar/squad_v2")

model_checkpoint = "deepset/roberta-base-squad2"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

print(datasets["train"][0])

model.safetensors: reconstructing file:   0%|          |  0.00B /  496MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'id': '56be85543aeaaa14008c9063', 'title': 'Beyoncé', 'context': 'Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny\'s Child. Managed by her father, Mathew Knowles, the group became one of the world\'s best-selling girl groups of all time. Their hiatus saw the release of Beyoncé\'s debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".', 'question': 'When did Beyonce start becoming popular?', 'answers': {'text': ['in the late 1990s'], 'answer_start': [269]}}


Paso 3: Preprocesamiento Inteligente (Mapeo de Tokens)

Este es el paso más técnico e importante del Procesamiento de Lenguaje Natural para tareas de Question Answering.

El modelo neuronal no genera palabras desde cero; su trabajo es comportarse como un resaltador. Necesita aprender a señalar en qué número de token exacto empieza la respuesta y en cuál termina. Como a veces los textos son muy largos y debemos recortarlos en pedazos (ventanas), necesitamos un algoritmo que recalcule matemáticamente dónde quedó la respuesta después del recorte.


In [3]:
max_length = 384
doc_stride = 128
# Verificamos de qué lado aplica el relleno (padding) nuestro modelo
pad_on_right = tokenizer.padding_side == "right"

def prepare_train_features(examples):
    # 1. Limpiamos espacios en blanco accidentales a la izquierda de las preguntas
    examples["question"] = [q.lstrip() for q in examples["question"]]

    # 2. Tokenización avanzada con truncamiento inteligente
    tokenized_examples = tokenizer(
        examples["question" if pad_on_right else "context"],
        examples["context" if pad_on_right else "question"],
        truncation="only_second" if pad_on_right else "only_first",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    # 3. Extracción de mapas para ubicar la respuesta
    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")

    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []

    # 4. Cálculo de coordenadas (inicio y fin) para cada fragmento de texto
    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = tokenized_examples.sequence_ids(i)

        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]

        # Si la respuesta no está en este fragmento, apuntamos al token [CLS] (vacío)
        if len(answers["answer_start"]) == 0:
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
        else:
            # Coordenadas en caracteres (texto original)
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])

            # Buscamos dónde empieza y termina el contexto dentro del array de tokens
            token_start_index = 0
            while sequence_ids[token_start_index] != (1 if pad_on_right else 0):
                token_start_index += 1
            token_end_index = len(input_ids) - 1
            while sequence_ids[token_end_index] != (1 if pad_on_right else 0):
                token_end_index -= 1

            # Verificamos si la respuesta quedó fuera de nuestro recorte (stride)
            if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
                tokenized_examples["start_positions"].append(cls_index)
                tokenized_examples["end_positions"].append(cls_index)
            else:
                # Movemos los índices hasta encontrar el token exacto donde inicia y termina la respuesta
                while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                    token_start_index += 1
                tokenized_examples["start_positions"].append(token_start_index - 1)

                while offsets[token_end_index][1] >= end_char:
                    token_end_index -= 1
                tokenized_examples["end_positions"].append(token_end_index + 1)

    return tokenized_examples

# 5. Aplicamos la función a todo el dataset de forma masiva
tokenized_datasets = datasets.map(prepare_train_features, batched=True, remove_columns=datasets["train"].column_names)
print("¡Preprocesamiento completado exitosamente y listo para entrenar!")

Map:   0%|          | 0/130319 [00:00<?, ? examples/s]

Map:   0%|          | 0/11873 [00:00<?, ? examples/s]

¡Preprocesamiento completado exitosamente y listo para entrenar!


4: El Entrenamiento (Fine-Tuning).


In [6]:
from transformers import TrainingArguments, Trainer

# 1. Definimos los hiperparámetros (las reglas del entrenamiento)
args = TrainingArguments(
    output_dir="./modelo_qa_oma",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,
)

# 2. Ensamblamos el motor de entrenamiento
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"].select(range(1000)),
    eval_dataset=tokenized_datasets["validation"].select(range(200)),
    processing_class=tokenizer,   # <-- ¡Aquí aplicamos la corrección moderna!
)

# 3. ¡Iniciamos el entrenamiento en la GPU!
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,2.040736
2,No log,1.698883
3,No log,1.863744


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=189, training_loss=0.3989426143585689, metrics={'train_runtime': 80.0958, 'train_samples_per_second': 37.455, 'train_steps_per_second': 2.36, 'total_flos': 587917702656000.0, 'train_loss': 0.3989426143585689, 'epoch': 3.0})

Paso 5: Puesta en Producción (Inferencia)

In [9]:
import torch

# 1. Definimos la base de conocimiento y la pregunta
contexto_operativo = """
El nuevo modelo operativo para las tiendas Café Oma en el tercer trimestre establece
que el presupuesto mensual de ventas para la sucursal Centro es de $45,000,000 COP.
Para lograr esta meta, la estrategia principal se enfocará en la distribución del
presupuesto a través del aumento del ticket promedio mediante promociones cruzadas
en las horas pico de la mañana y auditorías rigurosas de inventario.
"""
pregunta = "¿Cuál es la meta de ventas para la sucursal Centro?"

# 2. Tokenizamos la entrada (convertimos texto a números)
# return_tensors="pt" le dice que use el formato de PyTorch
inputs = tokenizer(pregunta, contexto_operativo, return_tensors="pt").to("cuda")

# 3. Pasamos los datos por nuestro modelo entrenado
model.eval() # Ponemos el modelo en modo evaluación
with torch.no_grad(): # Apagamos el cálculo de gradientes para ahorrar memoria (ya no estamos entrenando)
    outputs = model(**inputs)

# 4. Encontramos las coordenadas de la respuesta
# El modelo nos devuelve 'logits' (probabilidades matemáticas brutas) para cada token.
# torch.argmax busca el número de token con la probabilidad más alta de ser el inicio y el fin.
start_index = torch.argmax(outputs.start_logits)
end_index = torch.argmax(outputs.end_logits)

# 5. Decodificamos los tokens ganadores de vuelta a texto humano
predict_answer_tokens = inputs.input_ids[0, start_index : end_index + 1]
respuesta_final = tokenizer.decode(predict_answer_tokens, skip_special_tokens=True)

# 6. Imprimimos los resultados
print("ANÁLISIS DE TEXTO COMPLETADO (Método Nativo PyTorch)")
print("-" * 50)
print(f"Pregunta: {pregunta}")
print(f"Respuesta extraída: '{respuesta_final}'")

ANÁLISIS DE TEXTO COMPLETADO (Método Nativo PyTorch)
--------------------------------------------------
Pregunta: ¿Cuál es la meta de ventas para la sucursal Centro?
Respuesta extraída: ' $45,000,000 COP'


In [11]:
import torch
import time
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

# Definimos el diccionario con los tres modelos a comparar
modelos_hf = {
    "BERT Large (Clásico/Pesado)": "bert-large-uncased-whole-word-masking-finetuned-squad",
    "RoBERTa Base (Moderno/Equilibrado)": "deepset/roberta-base-squad2",
    "DistilBERT (Destilado/Rápido)": "distilbert-base-cased-distilled-squad"
}

contexto = """
Restcafé S.A.S. ha implementado un nuevo modelo de distribución presupuestal para las tiendas Café Oma.
El presupuesto mensual para la sucursal Norte es de $45,000,000, enfocado principalmente en aumentar
el ticket promedio a través de estrategias de up-selling en bebidas de temporada.
"""
pregunta = "¿De cuánto es el presupuesto mensual para la sucursal Norte?"

print("EXPERIMENTO: COMPARATIVA DE MODELOS HUGGING FACE (MÉTODO NATIVO PYTORCH)")
print("=" * 70)

# Iteramos sobre cada modelo para evaluar su rendimiento
for nombre, ruta in modelos_hf.items():
    print(f"\nCargando {nombre}...")

    # 1. Cargamos el tokenizador y el modelo matemático directamente a la GPU
    tokenizer_cmp = AutoTokenizer.from_pretrained(ruta)
    model_cmp = AutoModelForQuestionAnswering.from_pretrained(ruta).to("cuda")
    model_cmp.eval() # Modo evaluación

    # 2. Preparamos los tensores de entrada
    inputs_cmp = tokenizer_cmp(pregunta, contexto, return_tensors="pt").to("cuda")

    # 3. Medimos el tiempo exacto de inferencia neuronal
    inicio = time.time()
    with torch.no_grad():
        outputs_cmp = model_cmp(**inputs_cmp)

    # Buscamos las coordenadas con mayor probabilidad (logits)
    start_index = torch.argmax(outputs_cmp.start_logits)
    end_index = torch.argmax(outputs_cmp.end_logits)
    fin = time.time()

    # 4. Decodificamos la respuesta a texto humano
    predict_answer_tokens = inputs_cmp.input_ids[0, start_index : end_index + 1]
    respuesta = tokenizer_cmp.decode(predict_answer_tokens, skip_special_tokens=True)

    # 5. Mostramos las métricas
    print(f"  Respuesta extraída : '{respuesta}'")
    print(f"  Tiempo inferencia  : {(fin - inicio):.4f} segundos")
    print("-" * 70)

EXPERIMENTO: COMPARATIVA DE MODELOS HUGGING FACE (MÉTODO NATIVO PYTORCH)

Cargando BERT Large (Clásico/Pesado)...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.34GB            

...

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  261MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

  Respuesta extraída : '$ 45, 000, 000'
  Tiempo inferencia  : 0.0069 segundos
----------------------------------------------------------------------
